In [2]:
# Step 1 — Load Saved Models

import pandas as pd
import numpy as np
import joblib
import warnings

from sklearn.base import BaseEstimator, RegressorMixin

warnings.filterwarnings("ignore")


# --------------------------------------------------
# Compatibility class used by the saved
# Conversion Rate model
# --------------------------------------------------

class NonNegativeRegressor(BaseEstimator, RegressorMixin):

    def __init__(self, model):
        self.model = model

    def fit(self, X, y):
        self.model.fit(X, y)
        self.is_fitted_ = True
        return self

    def predict(self, X):
        predictions = self.model.predict(X)
        return np.maximum(predictions, 0)


# --------------------------------------------------
# Load trained models
# --------------------------------------------------

ctr_model = joblib.load(
    "../models/ctr_best_model.pkl"
)

conversion_rate_model = joblib.load(
    "../models/conversion_rate_best_model.pkl"
)

revenue_model = joblib.load(
    "../models/revenue_best_model.pkl"
)

profitability_model = joblib.load(
    "../models/profitability_best_model.pkl"
)


# --------------------------------------------------
# Confirmation
# --------------------------------------------------

print("All models loaded successfully.")

print("\nLoaded Models")
print("-" * 40)
print("1. CTR Model")
print("2. Conversion Rate Model")
print("3. Revenue Model")
print("4. Profitability Classification Model")

All models loaded successfully.

Loaded Models
----------------------------------------
1. CTR Model
2. Conversion Rate Model
3. Revenue Model
4. Profitability Classification Model


In [3]:
# Step 2 — Define Campaign Input Features

# --------------------------------------------------
# Categorical features
# --------------------------------------------------

categorical_features = [
    "campaign_objective",
    "platform",
    "device_type",
    "creative_format",
    "ad_copy_length",
    "income_bracket",
    "purchase_intent_score",
    "day_of_week",
    "industry_vertical"
]

# --------------------------------------------------
# Numerical features
# --------------------------------------------------

numerical_features = [
    "has_call_to_action",
    "retargeting_flag",
    "quarter",
    "campaign_day",
    "quality_score",
    "ad_spend"
]

# Combined feature list
model_features = (
    categorical_features +
    numerical_features
)

print("Recommendation Engine - Campaign Features")
print("-" * 50)

print("Total features:", len(model_features))
print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))

print("\nFeatures:")
for i, feature in enumerate(model_features, start=1):
    print(f"{i:2}. {feature}")

Recommendation Engine - Campaign Features
--------------------------------------------------
Total features: 15
Categorical features: 9
Numerical features: 6

Features:
 1. campaign_objective
 2. platform
 3. device_type
 4. creative_format
 5. ad_copy_length
 6. income_bracket
 7. purchase_intent_score
 8. day_of_week
 9. industry_vertical
10. has_call_to_action
11. retargeting_flag
12. quarter
13. campaign_day
14. quality_score
15. ad_spend


In [4]:
# Step 3 — Create Sample Campaign Input

sample_campaign = pd.DataFrame([{
    # Categorical features
    "campaign_objective": "Sales",
    "platform": "Google",
    "device_type": "Mobile",
    "creative_format": "Video",
    "ad_copy_length": "Medium",
    "income_bracket": "Medium",
    "purchase_intent_score": "High",
    "day_of_week": "Monday",
    "industry_vertical": "Technology",

    # Numerical features
    "has_call_to_action": 1,
    "retargeting_flag": 1,
    "quarter": 3,
    "campaign_day": 15,
    "quality_score": 8,
    "ad_spend": 10000
}])

print("Sample Campaign Created")
print("-" * 40)

sample_campaign

Sample Campaign Created
----------------------------------------


,campaign_objective,platform,device_type,creative_format,ad_copy_length,income_bracket,purchase_intent_score,day_of_week,industry_vertical,has_call_to_action,retargeting_flag,quarter,campaign_day,quality_score,ad_spend
0,Sales,Google,Mobile,Video,Medium,Medium,High,Monday,Technology,1,1,3,15,8,10000


In [5]:
# Step 4 — Generate Predictions

# --------------------------------------------------
# CTR Prediction
# --------------------------------------------------

predicted_ctr = ctr_model.predict(
    sample_campaign
)[0]


# --------------------------------------------------
# Conversion Rate Prediction
# --------------------------------------------------

predicted_conversion_rate = conversion_rate_model.predict(
    sample_campaign
)[0]


# --------------------------------------------------
# Revenue Prediction
# --------------------------------------------------

predicted_revenue = revenue_model.predict(
    sample_campaign
)[0]


# --------------------------------------------------
# Profitability Prediction
# --------------------------------------------------

profitability_probability = (
    profitability_model.predict_proba(
        sample_campaign
    )[0, 1]
)

profitability_prediction = (
    profitability_model.predict(
        sample_campaign
    )[0]
)


# --------------------------------------------------
# Display Results
# --------------------------------------------------

print("Campaign Prediction Results")
print("=" * 50)

print(
    f"Predicted CTR             : {predicted_ctr:.2f}%"
)

print(
    f"Predicted Conversion Rate : "
    f"{predicted_conversion_rate:.2f}%"
)

print(
    f"Predicted Revenue         : "
    f"₹{predicted_revenue:,.2f}"
)

print(
    f"Probability of Profit     : "
    f"{profitability_probability * 100:.2f}%"
)

print(
    f"Profitability Prediction  : "
    f"{'Profitable' if profitability_prediction == 1 else 'Loss-making'}"
)

Campaign Prediction Results
Predicted CTR             : 4.35%
Predicted Conversion Rate : 5.80%
Predicted Revenue         : ₹52,687.89
Probability of Profit     : 99.56%
Profitability Prediction  : Profitable


In [6]:
# Step 5 — Calculate Financial Metrics

# Get planned campaign spend
campaign_ad_spend = sample_campaign["ad_spend"].iloc[0]

# Calculate estimated profit
estimated_profit = (
    predicted_revenue - campaign_ad_spend
)

# Calculate estimated ROAS
estimated_roas = (
    predicted_revenue / campaign_ad_spend
)

# --------------------------------------------------
# Display Financial Metrics
# --------------------------------------------------

print("Campaign Financial Analysis")
print("=" * 50)

print(
    f"Planned Ad Spend   : ₹{campaign_ad_spend:,.2f}"
)

print(
    f"Predicted Revenue  : ₹{predicted_revenue:,.2f}"
)

print(
    f"Estimated Profit   : ₹{estimated_profit:,.2f}"
)

print(
    f"Estimated ROAS     : {estimated_roas:.2f}x"
)

Campaign Financial Analysis
Planned Ad Spend   : ₹10,000.00
Predicted Revenue  : ₹52,687.89
Estimated Profit   : ₹42,687.89
Estimated ROAS     : 5.27x


In [7]:
# Step 6 — Generate Campaign Recommendation

# --------------------------------------------------
# Determine recommendation category
# --------------------------------------------------

if profitability_probability >= 0.70:

    recommendation = "HIGH POTENTIAL"
    recommendation_message = (
        "Proceed with the campaign. "
        "The campaign has a strong probability of being profitable."
    )

elif profitability_probability >= 0.40:

    recommendation = "MODERATE POTENTIAL"
    recommendation_message = (
        "Review the campaign before proceeding. "
        "There is moderate potential for profitability."
    )

else:

    recommendation = "LOW POTENTIAL"
    recommendation_message = (
        "Reconsider the campaign. "
        "The probability of profitability is relatively low."
    )


# --------------------------------------------------
# Display recommendation
# --------------------------------------------------

print("CAMPAIGN RECOMMENDATION")
print("=" * 55)

print(
    f"Probability of Profitability : "
    f"{profitability_probability * 100:.2f}%"
)

print(
    f"Estimated Profit             : "
    f"₹{estimated_profit:,.2f}"
)

print(
    f"Estimated ROAS               : "
    f"{estimated_roas:.2f}x"
)

print("\nRecommendation")
print("-" * 55)

print(recommendation)

print("\nDecision")
print("-" * 55)

print(recommendation_message)

CAMPAIGN RECOMMENDATION
Probability of Profitability : 99.56%
Estimated Profit             : ₹42,687.89
Estimated ROAS               : 5.27x

Recommendation
-------------------------------------------------------
HIGH POTENTIAL

Decision
-------------------------------------------------------
Proceed with the campaign. The campaign has a strong probability of being profitable.


In [8]:
# Step 7 — Create Prediction Summary

prediction_summary = pd.DataFrame({
    "Metric": [
        "Predicted CTR",
        "Predicted Conversion Rate",
        "Predicted Revenue",
        "Planned Ad Spend",
        "Estimated Profit",
        "Estimated ROAS",
        "Probability of Profitability",
        "Profitability Prediction",
        "Final Recommendation"
    ],
    
    "Prediction": [
        f"{predicted_ctr:.2f}%",
        f"{predicted_conversion_rate:.2f}%",
        f"₹{predicted_revenue:,.2f}",
        f"₹{campaign_ad_spend:,.2f}",
        f"₹{estimated_profit:,.2f}",
        f"{estimated_roas:.2f}x",
        f"{profitability_probability * 100:.2f}%",
        (
            "Profitable"
            if profitability_prediction == 1
            else "Loss-making"
        ),
        recommendation
    ]
})

prediction_summary

,Metric,Prediction
0,Predicted CTR,4.35%
1,Predicted Conversion Rate,5.80%
2,Predicted Revenue,"₹52,687.89"
3,Planned Ad Spend,"₹10,000.00"
4,Estimated Profit,"₹42,687.89"
5,Estimated ROAS,5.27x
6,Probability of Profitability,99.56%
7,Profitability Prediction,Profitable
8,Final Recommendation,HIGH POTENTIAL


In [9]:
# Step 8 — Reusable Campaign Prediction Function

def predict_campaign(campaign_input):

    # ----------------------------------------------
    # Convert input into DataFrame
    # ----------------------------------------------

    campaign_data = pd.DataFrame([campaign_input])

    # ----------------------------------------------
    # Generate predictions
    # ----------------------------------------------

    predicted_ctr = ctr_model.predict(
        campaign_data
    )[0]

    predicted_conversion_rate = (
        conversion_rate_model.predict(
            campaign_data
        )[0]
    )

    predicted_revenue = revenue_model.predict(
        campaign_data
    )[0]

    # ----------------------------------------------
    # Profitability probability
    # ----------------------------------------------

    profitability_probability = (
        profitability_model
        .predict_proba(campaign_data)[0, 1]
    )

    profitability_prediction = (
        profitability_model.predict(
            campaign_data
        )[0]
    )

    # ----------------------------------------------
    # Financial calculations
    # ----------------------------------------------

    ad_spend = campaign_data[
        "ad_spend"
    ].iloc[0]

    estimated_profit = (
        predicted_revenue - ad_spend
    )

    estimated_roas = (
        predicted_revenue / ad_spend
        if ad_spend > 0
        else 0
    )

    # ----------------------------------------------
    # Recommendation
    # ----------------------------------------------

    if profitability_probability >= 0.70:

        recommendation = "HIGH POTENTIAL"

    elif profitability_probability >= 0.40:

        recommendation = "MODERATE POTENTIAL"

    else:

        recommendation = "LOW POTENTIAL"

    # ----------------------------------------------
    # Return results
    # ----------------------------------------------

    return {
        "Predicted CTR (%)": predicted_ctr,
        "Predicted Conversion Rate (%)": predicted_conversion_rate,
        "Predicted Revenue": predicted_revenue,
        "Ad Spend": ad_spend,
        "Estimated Profit": estimated_profit,
        "Estimated ROAS": estimated_roas,
        "Probability of Profitability (%)":
            profitability_probability * 100,
        "Profitability Prediction":
            (
                "Profitable"
                if profitability_prediction == 1
                else "Loss-making"
            ),
        "Recommendation": recommendation
    }

In [10]:
# Test the reusable prediction function

campaign_result = predict_campaign(
    sample_campaign.iloc[0].to_dict()
)

print("Reusable Prediction Function Test")
print("=" * 55)

for metric, value in campaign_result.items():

    if isinstance(value, float):

        if "Revenue" in metric or "Profit" in metric or "Spend" in metric:
            print(f"{metric}: ₹{value:,.2f}")

        elif "ROAS" in metric:
            print(f"{metric}: {value:.2f}x")

        elif "(%)" in metric:
            print(f"{metric}: {value:.2f}%")

        else:
            print(f"{metric}: {value:.2f}")

    else:
        print(f"{metric}: {value}")

Reusable Prediction Function Test
Predicted CTR (%): 4.35%
Predicted Conversion Rate (%): 5.80%
Predicted Revenue: ₹52,687.89
Ad Spend: 10000
Estimated Profit: ₹42,687.89
Estimated ROAS: 5.27x
Probability of Profitability (%): ₹99.56
Profitability Prediction: Profitable
Recommendation: HIGH POTENTIAL


In [14]:
# Step 9 — Test Multiple Campaign Scenarios

# --------------------------------------------------
# Define campaign scenarios
# --------------------------------------------------

campaign_scenarios = {

    "Campaign A - High Potential": {
        "campaign_objective": "Sales",
        "platform": "Google",
        "device_type": "Mobile",
        "creative_format": "Video",
        "ad_copy_length": "Medium",
        "income_bracket": "Medium",
        "purchase_intent_score": "High",
        "day_of_week": "Monday",
        "industry_vertical": "Technology",
        "has_call_to_action": 1,
        "retargeting_flag": 1,
        "quarter": 3,
        "campaign_day": 15,
        "quality_score": 8,
        "ad_spend": 10000
    },

    "Campaign B - Moderate Potential": {
        "campaign_objective": "Brand Awareness",
        "platform": "Facebook",
        "device_type": "Desktop",
        "creative_format": "Image",
        "ad_copy_length": "Long",
        "income_bracket": "Medium",
        "purchase_intent_score": "Medium",
        "day_of_week": "Wednesday",
        "industry_vertical": "Retail",
        "has_call_to_action": 1,
        "retargeting_flag": 0,
        "quarter": 2,
        "campaign_day": 20,
        "quality_score": 6,
        "ad_spend": 15000
    },

    "Campaign C - Low Potential": {
        "campaign_objective": "Traffic",
        "platform": "Twitter",
        "device_type": "Desktop",
        "creative_format": "Image",
        "ad_copy_length": "Short",
        "income_bracket": "Low",
        "purchase_intent_score": "Low",
        "day_of_week": "Sunday",
        "industry_vertical": "Travel",
        "has_call_to_action": 0,
        "retargeting_flag": 0,
        "quarter": 1,
        "campaign_day": 5,
        "quality_score": 3,
        "ad_spend": 20000
    }
}


# --------------------------------------------------
# Generate predictions
# --------------------------------------------------

scenario_results = []

for campaign_name, campaign_input in campaign_scenarios.items():

    result = predict_campaign(campaign_input)

    scenario_results.append({
        "Campaign": campaign_name,
        "CTR (%)": result["Predicted CTR (%)"],
        "Conversion Rate (%)": result["Predicted Conversion Rate (%)"],
        "Predicted Revenue": result["Predicted Revenue"],
        "Ad Spend": result["Ad Spend"],
        "Estimated Profit": result["Estimated Profit"],
        "ROAS": result["Estimated ROAS"],
        "Profitability Probability (%)": result["Probability of Profitability (%)"],
        "Prediction": result["Profitability Prediction"],
        "Recommendation": result["Recommendation"]
    })


# --------------------------------------------------
# Create comparison DataFrame
# --------------------------------------------------

scenario_results_df = pd.DataFrame(scenario_results)


# --------------------------------------------------
# Format values for easier interpretation
# --------------------------------------------------

display_df = scenario_results_df.copy()

display_df["CTR (%)"] = display_df["CTR (%)"].map(
    lambda x: f"{x:.2f}%"
)

display_df["Conversion Rate (%)"] = display_df["Conversion Rate (%)"].map(
    lambda x: f"{x:.2f}%"
)

display_df["Predicted Revenue"] = display_df["Predicted Revenue"].map(
    lambda x: f"₹{x:,.2f}"
)

display_df["Ad Spend"] = display_df["Ad Spend"].map(
    lambda x: f"₹{x:,.2f}"
)

display_df["Estimated Profit"] = display_df["Estimated Profit"].map(
    lambda x: f"₹{x:,.2f}"
)

display_df["ROAS"] = display_df["ROAS"].map(
    lambda x: f"{x:.2f}x"
)

display_df["Profitability Probability (%)"] = (
    display_df["Profitability Probability (%)"].map(
        lambda x: f"{x:.2f}%"
    )
)


# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Campaign Scenario Comparison")
print("=" * 110)

display(display_df)

Campaign Scenario Comparison


,Campaign,CTR (%),Conversion Rate (%),Predicted Revenue,Ad Spend,Estimated Profit,ROAS,Profitability Probability (%),Prediction,Recommendation
0,Campaign A - High Potential,4.35%,5.80%,"₹52,687.89","₹10,000.00","₹42,687.89",5.27x,99.56%,Profitable,HIGH POTENTIAL
1,Campaign B - Moderate Potential,2.75%,3.06%,"₹34,091.00","₹15,000.00","₹19,091.00",2.27x,97.83%,Profitable,HIGH POTENTIAL
2,Campaign C - Low Potential,0.69%,0.65%,"₹18,293.68","₹20,000.00","₹-1,706.32",0.91x,59.94%,Profitable,LOW POTENTIAL


In [15]:
# Step 11 — Generate Recommendation Summary

recommendation_summary = []

for campaign_name, campaign_input in campaign_scenarios.items():

    result = predict_campaign(campaign_input)

    recommendation_summary.append({
        "Campaign": campaign_name,
        "Recommendation": result["Recommendation"],
        "Profitability Probability": (
            f'{result["Probability of Profitability (%)"]:.2f}%'
        ),
        "Predicted Revenue": (
            f'₹{result["Predicted Revenue"]:,.2f}'
        ),
        "Estimated Profit": (
            f'₹{result["Estimated Profit"]:,.2f}'
        ),
        "ROAS": (
            f'{result["Estimated ROAS"]:.2f}x'
        )
    })

recommendation_summary_df = pd.DataFrame(
    recommendation_summary
)

print("Marketing Campaign Recommendation Summary")
print("=" * 90)

display(recommendation_summary_df)

Marketing Campaign Recommendation Summary


,Campaign,Recommendation,Profitability Probability,Predicted Revenue,Estimated Profit,ROAS
0,Campaign A - High Potential,HIGH POTENTIAL,99.56%,"₹52,687.89","₹42,687.89",5.27x
1,Campaign B - Moderate Potential,HIGH POTENTIAL,97.83%,"₹34,091.00","₹19,091.00",2.27x
2,Campaign C - Low Potential,LOW POTENTIAL,59.94%,"₹18,293.68","₹-1,706.32",0.91x
